In [1]:
# Challenge 02 -- Inventory Health Report

# You are an SC Analyst with 50 SKU's across 4 product categories and 3 warehouses
# The inventory team needs a monthly inventory health report
# ID: stockout risks, classify SKU's by value, flag SKU'd eroding working capital

In [2]:
# Import libraries and create the supply chain dataset
import pandas as pd
import numpy as np
import plotly.express as px

np.random.seed(9)

# structure the data
skus = [f"SKU-{str(i).zfill(3)}" for i in range(1, 51)]
categories = ["Electronics", "Apparel", "Hardware", "Consumables"]
warehouses = ["East", "West", "Central"]

# randomize data with np.random() by defined categories
rows = []
for sku in skus:
    category = np.random.choice(categories)
    warehouse = np.random.choice(warehouses)
    unit_cost = round(np.random.uniform(5, 500), 2)
    lead_time = np.random.randint(3, 30)
    daily_demand = round(np.random.uniform(1, 50), 1)
    units_on_hand = np.random.randint(0, 500)
    reorder_point = round(daily_demand * lead_time * 1.2)
    monthly_demand = round(daily_demand * 30)

    rows.append({
        "sku_id": sku,
        "category": category,
        "warehouse": warehouse,
        "unit_cost": unit_cost,
        "units_on_hand": units_on_hand,
        "daily_demand": daily_demand,
        "monthly_demand": monthly_demand,
        "lead_time_days": lead_time,
        "reorder_point": reorder_point
    })

df = pd.DataFrame(rows)
print(df.shape)
print(df.head(10))

# view in data wrangler
df

(50, 9)
    sku_id     category warehouse  unit_cost  units_on_hand  daily_demand  \
0  SKU-001     Hardware      East     253.43            406           1.4   
1  SKU-002      Apparel      West     113.19            248          13.2   
2  SKU-003      Apparel      West      87.55            187          19.3   
3  SKU-004  Electronics   Central     351.06             12          45.0   
4  SKU-005     Hardware   Central     278.76            421          19.9   
5  SKU-006  Consumables   Central     321.59            447          47.1   
6  SKU-007  Electronics      West      56.71            492          26.5   
7  SKU-008     Hardware   Central     108.69            245          48.6   
8  SKU-009  Consumables      East     292.97            388          45.4   
9  SKU-010  Consumables      West      41.88            388          30.9   

   monthly_demand  lead_time_days  reorder_point  
0              42              25             42  
1             396              11         

,sku_id,category,warehouse,unit_cost,units_on_hand,daily_demand,monthly_demand,lead_time_days,reorder_point
0,SKU-001,Hardware,East,253.43,406,1.4,42,25,42
1,SKU-002,Apparel,West,113.19,248,13.2,396,11,174
2,SKU-003,Apparel,West,87.55,187,19.3,579,3,69
3,SKU-004,Electronics,Central,351.06,12,45.0,1350,13,702
4,SKU-005,Hardware,Central,278.76,421,19.9,597,27,645
5,SKU-006,Consumables,Central,321.59,447,47.1,1413,19,1074
6,SKU-007,Electronics,West,56.71,492,26.5,795,4,127
7,SKU-008,Hardware,Central,108.69,245,48.6,1458,21,1225
8,SKU-009,Consumables,East,292.97,388,45.4,1362,13,708
9,SKU-010,Consumables,West,41.88,388,30.9,927,22,816


In [3]:
# Part 1:
# Validate the data -- shape, nulls, duplicates, dtypes, and business logic

# check the shape of the data
# remember that df.shape() returns a tuple (50, 9) --> (rows, columns)
# df.shape[0] accesses the first element --> the rows --> 50
# df.shape[1] accesses the second element --> the columns --> 9
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

# check the data for null values
# begin by defining nulls
nulls = df.isnull().sum()
# write an f-string
print(f"\nNull values per column:\n{nulls}")
print(f"Total nulls: {nulls.sum()}") 

# check for duplicate values
dupes = df.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")

# review the dataframe types
print(f"\nColumn dtypes:\n{df.dtypes}")

# descriptive stats of the shape of the dataframe
print(f"\nDescriptive Statistics:\n {df.describe().round(2)}")

# business logic -- sanity check on daily versus monthly demand
logicone = df[df["daily_demand"] > df["monthly_demand"]]
print(f"\nBusiness Logic Violations (daily_demand > monthly_demand): {len(logicone)}")

# business logic -- sanity check on daily versus monthly demand
logictwo = df[df["reorder_point"] <= 0]
print(f"\nBusiness Logic Violations (reorder_point <= 0): {len(logictwo)}")

# business logic -- sanity check on daily versus monthly demand
logicthree = df[df["units_on_hand"] < 0]
print(f"\nBusiness Logic Violations (units_on_hand < 0): {len(logicthree)}")


Dataset shape: 50 rows x 9 columns

Null values per column:
sku_id            0
category          0
warehouse         0
unit_cost         0
units_on_hand     0
daily_demand      0
monthly_demand    0
lead_time_days    0
reorder_point     0
dtype: int64
Total nulls: 0

Duplicate rows: 0

Column dtypes:
sku_id                str
category              str
warehouse             str
unit_cost         float64
units_on_hand       int64
daily_demand      float64
monthly_demand      int64
lead_time_days      int64
reorder_point       int64
dtype: object

Descriptive Statistics:
        unit_cost  units_on_hand  daily_demand  monthly_demand  lead_time_days  \
count      50.00          50.00         50.00           50.00           50.00   
mean      261.96         249.34         28.63          859.02           15.58   
std       151.79         137.03         13.30          398.92            8.05   
min         6.31          12.00          1.40           42.00            3.00   
25%       120.67  

In [4]:
# Part 2: Inventory Value & ABC Classification
# inventory_value = units_on_hand x unit_cost
# classify the SKU's into ABC tiers w/pd.cut()
# A= top 20%, B= next 30%, C= bottom 50%

# assign the inventory_value column to the dataframe
df = (
    df
    .assign(
        inventory_value = lambda x: (x["units_on_hand"] * x["unit_cost"]).round(2),
        abc_tier = lambda x: pd.qcut(
            x["inventory_value"],
            q=[0, 0.50, 0.80, 1.0],
            labels=["C", "B", "A"]
        )
    )
    )

# view the datarame
print(df.head(n=10))

# show the count of total inventory_value by abc_tier
summary = (
    df
    .pivot_table(
        index="abc_tier",
        values=["sku_id", "inventory_value"],
        aggfunc={"sku_id": "count", "inventory_value":"sum"}
    )
    .round(2)
)

# print the summary dataframe
print(summary)

    sku_id     category warehouse  unit_cost  units_on_hand  daily_demand  \
0  SKU-001     Hardware      East     253.43            406           1.4   
1  SKU-002      Apparel      West     113.19            248          13.2   
2  SKU-003      Apparel      West      87.55            187          19.3   
3  SKU-004  Electronics   Central     351.06             12          45.0   
4  SKU-005     Hardware   Central     278.76            421          19.9   
5  SKU-006  Consumables   Central     321.59            447          47.1   
6  SKU-007  Electronics      West      56.71            492          26.5   
7  SKU-008     Hardware   Central     108.69            245          48.6   
8  SKU-009  Consumables      East     292.97            388          45.4   
9  SKU-010  Consumables      West      41.88            388          30.9   

   monthly_demand  lead_time_days  reorder_point  inventory_value abc_tier  
0              42              25             42        102892.58        B 

In [5]:
# Part 3 — Stockout risk analysis:
# A SKU is at stockout risk if units_on_hand is below its reorder_point. 

# Calculate:
# days_of_supply — how many days of stock remain: units_on_hand / daily_demand
# stockout_risk — boolean flag: True if units_on_hand < reorder_point
# Filter and display all at-risk SKUs sorted by days_of_supply ascending — most urgent first

df = (
    df
    .assign(
        days_of_supply = lambda x: (x["units_on_hand"] / x["daily_demand"]).round(2),
        stockout_risk = lambda x: x["units_on_hand"] < x["reorder_point"], # creates a boolean instead of x[x[] < x[]] which filters
    )
    .sort_values(by="days_of_supply", ascending=True)
)

# filter for the at_risk sku's
risky = (
    df[df["stockout_risk"] == True]
        .sort_values(by="days_of_supply", ascending=True)
)

# write an f-string
print(f"SKU's at stockout risk: {len(risky)}")
print(risky[["sku_id", "category", "warehouse", "units_on_hand", "reorder_point", "days_of_supply", "abc_tier"]])

SKU's at stockout risk: 31
     sku_id     category warehouse  units_on_hand  reorder_point  \
3   SKU-004  Electronics   Central             12            702   
40  SKU-041      Apparel   Central             12            533   
11  SKU-012  Consumables      West             46           1167   
30  SKU-031  Consumables   Central             86           1374   
10  SKU-011  Consumables      East            108            262   
18  SKU-019     Hardware      East             66            423   
23  SKU-024  Electronics      West            113            503   
42  SKU-043  Consumables   Central            140            326   
29  SKU-030      Apparel      West            117            741   
34  SKU-035     Hardware      West            164           1119   
37  SKU-038      Apparel   Central             75            155   
41  SKU-042      Apparel      East            131            936   
14  SKU-015     Hardware      East            203           1413   
44  SKU-045  Electron

In [6]:
# Part 4 — Warehouse summary:
# Build a warehouse-level summary using .pivot_table() showing:

# Total inventory value per warehouse per category
pivotone = (
    df
    .pivot_table(
        index="warehouse",
        columns="category",
        values="inventory_value",
        aggfunc="sum"
    )
)
# print pivotone
print(pivotone)


# Count of SKUs at stockout risk per warehouse
pivottwo = (
    df
    .pivot_table(
        index="warehouse",
        columns="stockout_risk",
        values="sku_id",
        aggfunc="count"
    )
)
# print pivottwo
print(pivottwo)

# Average days of supply per warehouse
pivotthree = (
    df
    .pivot_table(
        index="warehouse",
        values="days_of_supply",
        aggfunc="mean"
    )
)
# print pivotthree
print(pivotthree)



category     Apparel  Consumables  Electronics   Hardware
warehouse                                                
Central    474981.90    483950.19    228753.86  373076.15
East       250684.62    121705.40     66405.60  463459.41
West       313666.22     25423.22    129999.64  246059.13
stockout_risk  False  True 
warehouse                  
Central            9     14
East               5      6
West               5     11
           days_of_supply
warehouse                
Central         14.021304
East            48.486364
West             9.171875


In [7]:
# Part 5 — Working capital analysis:
# Identify SKUs that are tying up excessive working capital — 
# defined as SKUs where inventory_value exceeds 2 standard deviations above the mean inventory value. 
# Flag these as "high_capital" and show which category and warehouse they belong to.

df = (
    df
    .assign(
        high_capital=lambda x: x["inventory_value"] > (x["inventory_value"].mean() + 2 * x["inventory_value"].std())
    )
)
print(df)

# pivot-table
pivotfour = (
    df[df["high_capital"] == True] # filter for "high_capital" SKU's --> calculation is Boolean (T/F)
    .pivot_table(
        index="warehouse",
        columns="category",
        values="high_capital",
        aggfunc="count"
    )
    .fillna(0)
)

print(pivotfour)


     sku_id     category warehouse  unit_cost  units_on_hand  daily_demand  \
3   SKU-004  Electronics   Central     351.06             12          45.0   
40  SKU-041      Apparel   Central     154.91             12          29.6   
11  SKU-012  Consumables      West     199.43             46          46.3   
30  SKU-031  Consumables   Central     484.02             86          47.7   
10  SKU-011  Consumables      East      74.38            108          43.7   
18  SKU-019     Hardware      East     262.16             66          25.2   
23  SKU-024  Electronics      West     180.46            113          38.1   
42  SKU-043  Consumables   Central     331.77            140          45.3   
29  SKU-030      Apparel      West     367.62            117          34.3   
34  SKU-035     Hardware      West     206.56            164          44.4   
37  SKU-038      Apparel   Central     427.36             75          18.4   
41  SKU-042      Apparel      East     456.60            131    

In [10]:
# Part 6
# Build one Plotly chart for a weekly operations meeting that shows stockout risk and inventory value simultaneously. 
# The scatter plot approach works well here:

# review column names
df.columns

# build scatterplot
fig = px.scatter(
    df,
    x="days_of_supply",
    y="inventory_value",
    color="stockout_risk",
    size="inventory_value",
    hover_name="sku_id",
    text="sku_id",
    title="Inventory Health - Days of Supply vs. Inventory Value"

)
fig.show()
